# 學習測試(進階修改架構)3.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import time


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備：{device}")


In [ ]:
all_data = pd.DataFrame()
all_labels = []


In [ ]:
def calculate_head_torso_angle(x_head, y_head, x_shoulder, y_shoulder):
    delta_x = x_head - x_shoulder
    delta_y = y_head - y_shoulder
    angle_radians = math.atan2(delta_y, delta_x)
    return math.degrees(angle_radians)


In [ ]:
def calculate_angle_change(x1, y1, x2, y2, x3, y3):

    a_x = x2 - x1
    a_y = y2 - y1

    b_x = x3 - x2
    b_y = y3 - y2

    dot_product = a_x * b_x + a_y * b_y

    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)
    if magnitude_a * magnitude_b == 0:
        return 0

    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))

    return math.acos(cos_angle) * (180 / math.pi)


In [ ]:
def calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist):

    a_x = x_elbow - x_shoulder
    a_y = y_elbow - y_shoulder

    b_x = x_wrist - x_elbow
    b_y = y_wrist - y_elbow

    dot_product = a_x * b_x + a_y * b_y

    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)

    if magnitude_a == 0 or magnitude_b == 0:
        return 0


    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))


    return math.degrees(math.acos(cos_angle))


In [ ]:
def calculate_arm_angle_for_direction(df, directions):
    arm_angles = []
    for i in range(len(directions)):
        if directions[i] == "right":

            x_shoulder = df['kpt_13'].astype(float)[i]
            y_shoulder = df['kpt_14'].astype(float)[i]
            x_elbow = df['kpt_15'].astype(float)[i]
            y_elbow = df['kpt_16'].astype(float)[i]
            x_wrist = df['kpt_17'].astype(float)[i]
            y_wrist = df['kpt_18'].astype(float)[i]
        else:

            x_shoulder = df['kpt_6'].astype(float)[i]
            y_shoulder = df['kpt_7'].astype(float)[i]
            x_elbow = df['kpt_8'].astype(float)[i]
            y_elbow = df['kpt_9'].astype(float)[i]
            x_wrist = df['kpt_10'].astype(float)[i]
            y_wrist = df['kpt_11'].astype(float)[i]


        angle = calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist)
        arm_angles.append(angle)

    return arm_angles


In [ ]:
def determine_walking_direction(df):
    """
    根據肩膀的水平位置判斷病患是向左走還是向右走，並返回行走方向。
    """
    x_shoulder_left = df['kpt_5'].astype(float)
    x_shoulder_right = df['kpt_6'].astype(float)
    directions = []
    for i in range(len(x_shoulder_left)):
        if x_shoulder_left[i] < x_shoulder_right[i]:
            directions.append("left")
        else:
            directions.append("right")
    return directions


In [ ]:
def step_x(x1, x2):
    return abs(x2 - x1)


In [ ]:
def calculate_step_length_std(step_lengths):
    return np.std(step_lengths) if len(step_lengths) > 1 else 0


In [ ]:
def calculate_acceleration(speeds, time_series):
    speeds = np.array(speeds)
    time_series = np.array(time_series)


    time_diff = np.diff(time_series)
    if np.any(time_diff == 0):
        return [0] * len(speeds)

    acceleration = np.diff(speeds) / time_diff
    acceleration = list(acceleration) + [0]
    return acceleration


In [ ]:
def check_for_nan_in_features(feature_vector):

    for key, value in feature_vector.items():
        if np.isnan(value):
            print(f"{key} 含有 NaN 值：{value}")


def upload_data_and_label(label):
    global all_data, all_labels


    uploaded_files = files.upload()


    for file_name in uploaded_files.keys():
        df = pd.read_csv(io.BytesIO(uploaded_files[file_name]))


        if 'kpts' in df.columns:
            df['kpts'] = df['kpts'].str.replace('"', '').str.strip()
            kpts_columns = df['kpts'].str.split(',', expand=True)
            for i in range(kpts_columns.shape[1]):
                kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)
            df = df.join(kpts_columns, rsuffix='_kpts')
            for col in kpts_columns.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


        time_series = pd.Series(range(len(df)))


        head_torso_angles = []
        arm_angles = []
        step_lengths = []

        for i in range(len(df)):
            x_head = (df['kpt_1'].iloc[i] + df['kpt_2'].iloc[i]) / 2
            y_head = (df['kpt_3'].iloc[i] + df['kpt_4'].iloc[i]) / 2
            x_shoulder = (df['kpt_5'].iloc[i] + df['kpt_6'].iloc[i]) / 2
            y_shoulder = (df['kpt_7'].iloc[i] + df['kpt_8'].iloc[i]) / 2

            x_elbow = df['kpt_9'].iloc[i]
            y_elbow = df['kpt_10'].iloc[i]
            x_wrist = df['kpt_11'].iloc[i]
            y_wrist = df['kpt_12'].iloc[i]

            x_ankle_left = df['kpt_15'].iloc[i]
            x_ankle_right = df['kpt_16'].iloc[i]


            head_torso_angle = calculate_head_torso_angle(x_head, y_head, x_shoulder, y_shoulder)
            arm_angle = calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist)
            step_length = step_x(x_ankle_left, x_ankle_right)

            head_torso_angles.append(head_torso_angle)
            arm_angles.append(arm_angle)
            step_lengths.append(step_length)


        step_frequency = len(df) / (time_series.iloc[-1] - time_series.iloc[0] + 1)
        speeds = [step_lengths[i] / (time_series[i] + 1) for i in range(len(step_lengths))]
        accelerations = calculate_acceleration(speeds, time_series)
        step_length_std = calculate_step_length_std(step_lengths)


        feature_vector = {
            'head_torso_angle': np.mean(head_torso_angles),
            'arm_angle': np.mean(arm_angles),
            'step_length': np.mean(step_lengths),
            'step_length_std': step_length_std,
            'step_frequency': step_frequency,
            'acceleration': np.mean(accelerations)
        }


        check_for_nan_in_features(feature_vector)


        feature_df = pd.DataFrame([feature_vector])


        all_data = pd.concat([all_data, feature_df], ignore_index=True)
        all_labels.append(label)

        print(f"標籤為 {label} 的文件 {file_name} 已處理完成。")


In [ ]:
print("上傳第一批數據並指定標籤為1")
upload_data_and_label(1)

print("上傳第二批數據並指定標籤為2")
upload_data_and_label(2)

print("上傳第三批數據並指定標籤為3")
upload_data_and_label(3)

print("上傳第四批數據並指定標籤為4")
upload_data_and_label(4)


處理

In [ ]:
unique_labels = set(all_labels)
print(f"原始標籤的唯一值: {unique_labels}")


if min(unique_labels) == 1:
    all_labels = [label - 1 for label in all_labels]


In [ ]:
print(f"調整後的標籤範圍: {min(all_labels)} 到 {max(all_labels)}")


In [ ]:
features = all_data.values
labels = np.array(all_labels)


In [ ]:
data_with_labels = pd.DataFrame(features)
data_with_labels['label'] = labels


data_with_labels = data_with_labels[data_with_labels['label'] != -1]


features = data_with_labels.drop(columns=['label']).values
labels = data_with_labels['label'].values


print(f"清理後的標籤範圍: {min(labels)} 到 {max(labels)}")


In [ ]:
scaler = StandardScaler()
features = scaler.fit_transform(features)


監督式學習

In [ ]:
class SeverityClassifier(nn.Module):
    def __init__(self, input_size, num_classes=4):
        super(SeverityClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 16)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(16, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.relu3(out)
        out = self.fc4(out)
        return out


In [ ]:
class PatientDataset(Dataset):
    def __init__(self, data, labels):
        self.features = data
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


In [ ]:
features = all_data[['head_torso_angle', 'arm_angle', 'step_length', 'step_frequency', 'acceleration', 'step_length_std']].values
labels = np.array(all_labels)


In [ ]:
scaler = StandardScaler()
features = scaler.fit_transform(features)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42, stratify=labels)


In [ ]:
train_dataset = PatientDataset(X_train, y_train)
test_dataset = PatientDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
input_size = features.shape[1]
num_classes = 4
model = SeverityClassifier(input_size=input_size, num_classes=num_classes).to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


In [ ]:
def train_model(num_epochs=50):
    model.train()
    for epoch in range(num_epochs):
        for features_batch, labels_batch in train_loader:
            features_batch = features_batch.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model(features_batch)
            loss = criterion(outputs, labels_batch)

            if torch.isnan(loss):
                print(f"發現 NaN 損失於 Epoch {epoch+1}")
                print(f"特徵範圍: {features_batch.min()} ~ {features_batch.max()}")
                print(f"標籤: {labels_batch}")
                return

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


In [ ]:
def test_model():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'測試集上的準確率: {100 * correct / total:.2f}%')


In [ ]:
train_model(num_epochs=50)


test_model()


In [ ]:
print(f"標準化前的特徵最大值: {all_data.values.max()}, 最小值: {all_data.values.min()}")


nan_before_scaling = np.isnan(all_data.values).sum()
inf_before_scaling = np.isinf(all_data.values).sum()
print(f"標準化前的特徵中的 NaN 值數量: {nan_before_scaling}, 無限值數量: {inf_before_scaling}")


In [ ]:
unique_labels = np.unique(labels)
print(f"標籤的唯一值: {unique_labels}")


In [ ]:
def train_model(num_epochs=50):
    model.train()
    for epoch in range(num_epochs):
        for features_batch, labels_batch in train_loader:
            features_batch = features_batch.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model(features_batch)
            loss = criterion(outputs, labels_batch)

            if torch.isnan(loss):
                print(f"發現 NaN 損失於 Epoch {epoch+1}")
                print(f"特徵範圍: {features_batch.min()} ~ {features_batch.max()}")
                print(f"標籤: {labels_batch}")
                return

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


 訓練 SVC 和 KNeighborsClassifier

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42, stratify=labels)


In [ ]:
svc_model = SVC(kernel='linear')
svc_model.fit(X_train, y_train)


In [ ]:
svc_preds = svc_model.predict(X_test)


In [ ]:
svc_cm = confusion_matrix(y_test, svc_preds)


In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.heatmap(svc_cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('SVC Confusion Matrix')


In [ ]:
knc_model = KNeighborsClassifier(n_neighbors=5)
knc_model.fit(X_train, y_train)


In [ ]:
knc_preds = knc_model.predict(X_test)


In [ ]:
knc_cm = confusion_matrix(y_test, knc_preds)


In [ ]:
plt.subplot(1, 2, 2)
sns.heatmap(knc_cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('KNeighborsClassifier Confusion Matrix')


In [ ]:
plt.tight_layout()
plt.show()
